# Multi-seed replication

Re-runs the second-corpus and cross-dataset headline experiments across 5 seeds so the numbers carry cross-seed variance bars in addition to per-seed bootstrap CIs. Targets:

    A3 Lee 2019 cross-session re-ID (experiment 20)
    A4 Lee 2019 open-set verification within-session (experiment 24)
    A4 cross-dataset symmetric, all four directions (experiment 26)

Output: results/34_multi_seed.json with per-seed rows plus aggregated mean / std / n across seeds.

Prerequisite: `A3_lee2019_download.ipynb` must have completed.

**Runtime -> A100 strongly recommended.** Expected wall: ~5-7 h (~9 experiment runs at ~30-60 min each). On L4 this is 8-10 h, split across sessions.

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
os.environ['BCI_LEE2019_CACHE'] = '/content/drive/MyDrive/bci_cache'
import glob, sys
WIN = '/content/drive/MyDrive/bci_cache/lee2019_mi/windowed'
files = sorted(glob.glob(os.path.join(WIN, '*.npz')))
print(f'Cached: {len(files)}/108')
if len(files) < 108:
    sys.exit('!!! cache incomplete; run A3_lee2019_download.ipynb first.')

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## Run the 5-seed sweep (--full). If wall-time tight, swap to --quick (3 seeds).

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.34_multi_seed --full

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
if os.path.isdir('/content/bci-identity-leakage'): os.chdir('/content/bci-identity-leakage')
RESULTS = 'results/34_multi_seed.json'
TAG = 'multi_seed'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': 'experiments.34_multi_seed',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing.')
print('--- BEGIN 34_multi_seed.json ---')
print(open(RESULTS).read())
print('--- END 34_multi_seed.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')